In [ ]:
# 1. 安裝所需套件
!pip install gradio

# 2. 載入模型開發與影像處理套件
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.utils import to_categorical
from PIL import Image
import gradio as gr

# 3. 準備並處理 MNIST 訓練資料
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# 依照 DNN 的要求，拉平影像並做 0~1 的正規化
x_train = x_train.reshape(60000, 784) / 255.0
x_test = x_test.reshape(10000, 784) / 255.0

# 輸出轉為 One-hot encoding (10種分類)
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print("====== DNN：資料準備完成！======")


### 💡 終極優化：DNN 架構升級與「包圍盒置中預處理」
在使用「全連接神經網路 (DNN)」的前提下，我發現模型對手寫數字的「大小」與「位置」極度敏感。為了解決 DNN 缺乏空間平移容忍度的天生弱點，我進行了兩大改良：

1. **網路架構深化與特徵穩定**：
   - 擴充為 `512 -> 512 -> 256` 的三層深層結構。
   - 加入了現代深度學習強烈推薦的 `BatchNormalization (批次標準化)` 層，大幅穩定深層網路的梯度，讓原本容易發散的訓練過程收斂得更好。
   - 使用 `Adam` 優化器與 `categorical_crossentropy` 取代傳統的 SGD 與 mse。
2. **導入 MNIST 原廠級的「特徵自動置中」影像預處理**：
   - 使用 OpenCV 的 `boundingRect` 演算法，讓系統自動「切下真正有畫字的部分」，等比例放大後，**精準貼到畫布最中央**。這個演算法徹底消除了使用者手寫位置偏離造成的辨識錯誤。



In [ ]:
# 建立 DNN 模型
model = Sequential()

# 第 1 層隱藏層 (512個神經元 + 批次標準化 + Dropout)
model.add(Dense(512, input_dim=784, use_bias=False))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))

# 第 2 層隱藏層 (512個神經元)
model.add(Dense(512, use_bias=False))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))

# 第 3 層隱藏層 (256個神經元)
model.add(Dense(256, use_bias=False))
model.add(BatchNormalization())
model.add(Activation('relu'))

# 輸出層 (10種分類的機率)
model.add(Dense(10, activation='softmax'))

# 使用更好的 Adam 優化器與分類專用 Loss
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 自動顯示網路與開始訓練
model.summary()

print("開始訓練深層 DNN 模型...")
history = model.fit(x_train, y_train, batch_size=128, epochs=15, validation_data=(x_test, y_test))
print("模型訓練大功告成！")


### 📦 外部套件 Gradio 說明與展示
本程式結合了 `Gradio` 來產出 Web 測試介面。
Gradio 能夠在短短幾句程式碼內，架設起 `Sketchpad()` (手寫畫板) 並連接到後端的神經網路預測函數，省去了撰寫 HTML/JS 前端以及建立 Flask API 的麻煩，是迅速展示 AI 模型成果的終極兵器。


In [ ]:
def resize_image_dnn_advanced(inp):
    # 1. 將 Gradio 圖層轉換成背景為白色的影像
    image = np.array(inp["layers"][0], dtype=np.float32).astype(np.uint8)
    image_pil = Image.fromarray(image)
    background = Image.new("RGB", image_pil.size, (255, 255, 255))
    background.paste(image_pil, mask=image_pil.split()[3])

    # 轉換為黑底白字
    image_gray = background.convert("L")
    img_array = 255 - np.array(image_gray)

    # 2. 自動鎖定「畫筆邊界框 (Bounding Box)」
    coords = cv2.findNonZero(img_array)
    if coords is not None:
        # 切下字跡的部分
        x, y, w, h = cv2.boundingRect(coords)
        cropped_digit = img_array[y:y+h, x:x+w]

        # 3. 把字跡長邊縮放成 20 像素 (還原 MNIST 訓練集的預設做法)
        if w > 0 and h > 0:
            if w > h:
                new_w = 20
                new_h = max(1, int(h * (20.0 / w)))
            else:
                new_h = 20
                new_w = max(1, int(w * (20.0 / h)))
            resized_digit = cv2.resize(cropped_digit, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)
        else:
            resized_digit = cropped_digit
            new_w, new_h = w, h

        # 4. 把縮放後的字跡完美地貼到 28x28 全黑畫布的「正中央」
        final_img = np.zeros((28, 28), dtype=np.uint8)
        off_x = (28 - new_w) // 2
        off_y = (28 - new_h) // 2
        final_img[off_y:off_y+new_h, off_x:off_x+new_w] = resized_digit
    else:
        final_img = np.zeros((28, 28), dtype=np.uint8)

    # 5. 拉平成 784，符合 DNN 需要的 1D 格式
    final_input = final_img.reshape(1, 784) / 255.0
    return final_input

def recognize_digit_dnn(inp):
    try:
        # 將畫板影像送入處理常式
        img_array = resize_image_dnn_advanced(inp)
        # 用升級版的 DNN 預測
        prediction = model.predict(img_array).flatten()
        labels = list('0123456789')
        return {labels[i]: float(prediction[i]) for i in range(10)}
    except Exception as e:
        return {"發生錯誤，請重畫": 0.0}

# 啟動 Gradio (拿掉不支援的畫筆粗細參數，恢復原廠設定)
iface = gr.Interface(
    fn=recognize_digit_dnn,
    inputs=gr.Sketchpad(),
    outputs=gr.Label(num_top_classes=3),
    title="😎",
    description="我透過『邊界框自動對齊 (Bounding Box)』技術消除誤差。現在就算特別把字畫在角落，它也能幫我抓回中間給模型測！(但還是請盡量稍微畫粗一點喔！)"
)

iface.launch(share=True, debug=True)
